# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [3]:
# ============================================================
# 1. RANKED ACTIONS + REASON CODES
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

FEATURES = [
    "clicks_7d_avg",
    "impressions_7d_avg",
    "position_7d_avg",
    "ctr_7d",
    "is_weekend"
]

TARGET = "target_next_day_clicks"

# Use the same processed feature dataset created in earlier weeks.
FEATURE_PATH = "refresh_feature_vector.csv"

if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(
        f"{FEATURE_PATH} was not found. "
        "Run the earlier feature-building notebook first."
    )

df = pd.read_csv(FEATURE_PATH)

df["report_date"] = pd.to_datetime(df["report_date"])

required = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    *FEATURES,
    TARGET
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# Clean data
# ------------------------------------------------------------

df = df.dropna(subset=[TARGET]).copy()

for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[FEATURES] = df[FEATURES].fillna(0)

# ------------------------------------------------------------
# Time-aware training split
# Same basic design used in W05
# ------------------------------------------------------------

dates = np.sort(df["report_date"].dropna().unique())

cutoff = dates[int(len(dates) * 0.80)]

train_df = df[df["report_date"] < cutoff].copy()
queue_df = df[df["report_date"] >= cutoff].copy()

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(float)

# ------------------------------------------------------------
# Train the W05 Gradient Boosting model
# ------------------------------------------------------------

action_model = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=20,
    random_state=42
)

action_model.fit(X_train, y_train)

queue_df["predicted_next_day_clicks"] = np.clip(
    action_model.predict(queue_df[FEATURES]),
    0,
    None
)

# ------------------------------------------------------------
# Reason codes
# ------------------------------------------------------------

# Use training quantiles as simple, data-driven thresholds.
ctr_low = train_df["ctr_7d"].quantile(0.25)
position_high = train_df["position_7d_avg"].quantile(0.75)
impressions_high = train_df["impressions_7d_avg"].quantile(0.75)
clicks_low = train_df["clicks_7d_avg"].quantile(0.25)

def make_reason(row):

    reasons = []

    if row["ctr_7d"] <= ctr_low:
        reasons.append("LOW_CTR")

    if row["position_7d_avg"] >= position_high:
        reasons.append("WEAKER_POSITION")

    if row["impressions_7d_avg"] >= impressions_high:
        reasons.append("HIGH_IMPRESSIONS")

    if row["clicks_7d_avg"] <= clicks_low:
        reasons.append("LOW_RECENT_CLICKS")

    if not reasons:
        reasons.append("MODEL_PRIORITY")

    return ";".join(reasons[:3])


queue_df["reason_code"] = queue_df.apply(
    make_reason,
    axis=1
)

# ------------------------------------------------------------
# Action recommendation
# ------------------------------------------------------------

def choose_action(row):

    if "LOW_CTR" in row["reason_code"]:
        return "Review title/snippet alignment and search intent"

    if "WEAKER_POSITION" in row["reason_code"]:
        return "Review content relevance and on-page quality"

    if "HIGH_IMPRESSIONS" in row["reason_code"]:
        return "Review whether the page is converting available visibility"

    if "LOW_RECENT_CLICKS" in row["reason_code"]:
        return "Review content relevance and click opportunity"

    return "Human review of content opportunity"


queue_df["recommended_action"] = queue_df.apply(
    choose_action,
    axis=1
)

# ------------------------------------------------------------
# Rank
# ------------------------------------------------------------

queue_df = queue_df.sort_values(
    "predicted_next_day_clicks",
    ascending=False
).reset_index(drop=True)

queue_df["priority_rank"] = np.arange(1, len(queue_df) + 1)

# Keep a manageable action queue.
ACTION_QUEUE_SIZE = min(1000, len(queue_df))

action_queue = queue_df.head(ACTION_QUEUE_SIZE).copy()

ACTION_COLUMNS = [
    "priority_rank",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "predicted_next_day_clicks",
    "clicks_7d_avg",
    "impressions_7d_avg",
    "position_7d_avg",
    "ctr_7d",
    "reason_code",
    "recommended_action"
]

action_queue = action_queue[ACTION_COLUMNS]

display(action_queue.head(20))

print("Action queue created.")
print("Rows:", len(action_queue))
print("Training cutoff:", pd.Timestamp(cutoff).date())

,priority_rank,client_hash_id,content_hash_id,report_date,predicted_next_day_clicks,clicks_7d_avg,impressions_7d_avg,position_7d_avg,ctr_7d,reason_code,recommended_action
0,1,client_20259bd6705d81d4,content_0ec90963d98b97a5,2026-03-02,57.876558,82.5,5304.0,3.788900,0.015554,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
1,2,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,2026-03-02,57.876558,104.5,6398.5,2.772111,0.016332,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
2,3,client_73cda7b4e4f265ea,content_471d9cabce329a66,2026-03-02,52.568108,32.0,5880.0,2.939292,0.005442,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
3,4,client_23a62021009f63c4,content_e8a52cf3d5988c07,2026-03-02,52.227578,32.0,7844.5,15.553710,0.004079,WEAKER_POSITION;HIGH_IMPRESSIONS,Review content relevance and on-page quality
4,5,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,2026-03-02,52.066905,64.5,9035.5,3.848911,0.007139,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
5,6,client_73cda7b4e4f265ea,content_e762b3812901065c,2026-03-02,52.066905,32.0,4006.0,3.036200,0.007988,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
6,7,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-02,47.711911,64.0,5355.5,2.741675,0.011950,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
7,8,client_e547b89c05043229,content_c9a0c2fdbdbfb562,2026-03-02,47.408685,52.0,4583.0,1.722492,0.011346,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
8,9,client_e547b89c05043229,content_ec2e0346994fb5a5,2026-03-02,46.997079,71.0,14181.0,2.073415,0.005007,HIGH_IMPRESSIONS,Review whether the page is converting availabl...
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,2026-03-02,46.722632,43.5,7089.5,2.635653,0.006136,HIGH_IMPRESSIONS,Review whether the page is converting availabl...


Action queue created.
Rows: 1000
Training cutoff: 2026-03-02


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [4]:
# ============================================================
# 2. INTENDED USE AND LIMITS
# ============================================================

intended_use = {
    "primary_user": "SEO/content analyst",
    "purpose": (
        "Prioritize content items for human review using observed "
        "features and measured model predictions."
    ),
    "decision_type": "Decision-support",
    "prediction_target": "Next-day clicks",
    "validation_design": "Time-aware evaluation",
    "recommended_use": [
        "Prioritize which content items deserve review first",
        "Identify observable patterns associated with model priority",
        "Support analyst investigation",
        "Create a repeatable review queue"
    ],
    "limits": [
        "Predictions are not causal recommendations",
        "The model does not prove that an action will increase clicks",
        "Results are directional and depend on the evaluated data",
        "Performance may change on new time periods or clients",
        "Human review is required before making content changes"
    ]
}

for key, value in intended_use.items():

    print(f"\n{key.upper()}")
    print("-" * len(key))

    if isinstance(value, list):
        for item in value:
            print(f"- {item}")
    else:
        print(value)


PRIMARY_USER
------------
SEO/content analyst

PURPOSE
-------
Prioritize content items for human review using observed features and measured model predictions.

DECISION_TYPE
-------------
Decision-support

PREDICTION_TARGET
-----------------
Next-day clicks

VALIDATION_DESIGN
-----------------
Time-aware evaluation

RECOMMENDED_USE
---------------
- Prioritize which content items deserve review first
- Identify observable patterns associated with model priority
- Support analyst investigation
- Create a repeatable review queue

LIMITS
------
- Predictions are not causal recommendations
- The model does not prove that an action will increase clicks
- Results are directional and depend on the evaluated data
- Performance may change on new time periods or clients
- Human review is required before making content changes


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [5]:
# ============================================================
# 3. HUMAN REVIEW + NO-GO LIST
# ============================================================

human_review_checklist = [
    "Check the current page and search intent before making changes.",
    "Check whether the observed CTR/position pattern is meaningful.",
    "Check recent performance rather than relying only on one prediction.",
    "Confirm that the page actually needs an intervention.",
    "Review title, snippet, content relevance and page quality manually.",
    "Record the reason for any action taken.",
    "Compare the result after the change using a defined monitoring period."
]

no_go_list = [
    "Do not automatically publish or edit content.",
    "Do not automatically change titles or metadata.",
    "Do not claim that a recommended action causes more clicks.",
    "Do not treat model ranking as proof of search-engine causality.",
    "Do not make decisions using future information.",
    "Do not use the model when required input data is missing or stale.",
    "Do not use the queue as a replacement for expert review."
]

print("HUMAN REVIEW CHECKLIST")
print("======================")

for i, item in enumerate(human_review_checklist, 1):
    print(f"{i}. {item}")

print("\nNO-GO LIST")
print("==========")

for i, item in enumerate(no_go_list, 1):
    print(f"{i}. {item}")

HUMAN REVIEW CHECKLIST
1. Check the current page and search intent before making changes.
2. Check whether the observed CTR/position pattern is meaningful.
3. Check recent performance rather than relying only on one prediction.
4. Confirm that the page actually needs an intervention.
5. Review title, snippet, content relevance and page quality manually.
6. Record the reason for any action taken.
7. Compare the result after the change using a defined monitoring period.

NO-GO LIST
1. Do not automatically publish or edit content.
2. Do not automatically change titles or metadata.
3. Do not claim that a recommended action causes more clicks.
4. Do not treat model ranking as proof of search-engine causality.
5. Do not make decisions using future information.
6. Do not use the model when required input data is missing or stale.
7. Do not use the queue as a replacement for expert review.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [6]:
# ============================================================
# 4. MONITORING / RETRAIN TRIGGERS
# ============================================================

monitoring_rules = pd.DataFrame([
    {
        "signal": "Prediction error",
        "metric": "MAE / RMSE",
        "trigger": "Error increases materially versus the validation benchmark",
        "response": "Investigate data drift and retrain if sustained"
    },
    {
        "signal": "Feature drift",
        "metric": "Feature distribution",
        "trigger": "Current feature distributions differ substantially from training",
        "response": "Review data pipeline and retrain if needed"
    },
    {
        "signal": "CTR drift",
        "metric": "ctr_7d",
        "trigger": "CTR distribution changes materially",
        "response": "Review whether the model still represents current behavior"
    },
    {
        "signal": "Position drift",
        "metric": "position_7d_avg",
        "trigger": "Position distribution changes materially",
        "response": "Reassess model performance"
    },
    {
        "signal": "Missing data",
        "metric": "Feature availability",
        "trigger": "Required features become unavailable or incomplete",
        "response": "Pause recommendations until the data issue is resolved"
    },
    {
        "signal": "Time drift",
        "metric": "Recent-period performance",
        "trigger": "Recent performance falls below the validated level",
        "response": "Run a fresh time-aware validation"
    }
])

display(monitoring_rules)

# ------------------------------------------------------------
# Optional helper for comparing current data with training
# ------------------------------------------------------------

feature_drift_summary = []

for feature in FEATURES:

    train_mean = train_df[feature].mean()
    current_mean = queue_df[feature].mean()

    if train_mean != 0:
        relative_change = (
            (current_mean - train_mean) /
            abs(train_mean)
        ) * 100
    else:
        relative_change = np.nan

    feature_drift_summary.append({
        "feature": feature,
        "training_mean": train_mean,
        "current_mean": current_mean,
        "relative_change_percent": relative_change
    })

feature_drift_summary = pd.DataFrame(feature_drift_summary)

display(feature_drift_summary)

,signal,metric,trigger,response
0,Prediction error,MAE / RMSE,Error increases materially versus the validati...,Investigate data drift and retrain if sustained
1,Feature drift,Feature distribution,Current feature distributions differ substanti...,Review data pipeline and retrain if needed
2,CTR drift,ctr_7d,CTR distribution changes materially,Review whether the model still represents curr...
3,Position drift,position_7d_avg,Position distribution changes materially,Reassess model performance
4,Missing data,Feature availability,Required features become unavailable or incomp...,Pause recommendations until the data issue is ...
5,Time drift,Recent-period performance,Recent performance falls below the validated l...,Run a fresh time-aware validation


,feature,training_mean,current_mean,relative_change_percent
0,clicks_7d_avg,0.238151,0.249926,4.944379
1,impressions_7d_avg,75.623521,77.919592,3.036186
2,position_7d_avg,12.031932,12.254436,1.849276
3,ctr_7d,0.003396,0.003615,6.447855
4,is_weekend,1.000000,0.000000,-100.000000


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
# ============================================================
# 5. EXPORTS FOR THE PAPER
# ============================================================

from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Export ranked action queue
# ------------------------------------------------------------

queue_path = OUTPUT_DIR / "content_action_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)

print(f"Saved action queue: {queue_path}")

# ------------------------------------------------------------
# Export monitoring rules
# ------------------------------------------------------------

monitoring_path = OUTPUT_DIR / "monitoring_retrain_triggers.csv"

monitoring_rules.to_csv(
    monitoring_path,
    index=False
)

print(f"Saved monitoring rules: {monitoring_path}")

# ------------------------------------------------------------
# Export feature drift summary
# ------------------------------------------------------------

drift_path = OUTPUT_DIR / "feature_drift_summary.csv"

feature_drift_summary.to_csv(
    drift_path,
    index=False
)

print(f"Saved feature drift summary: {drift_path}")

# ------------------------------------------------------------
# Export reason-code summary
# ------------------------------------------------------------

reason_summary = (
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

reason_summary_path = OUTPUT_DIR / "reason_code_summary.csv"

reason_summary.to_csv(
    reason_summary_path,
    index=False
)

print(f"Saved reason-code summary: {reason_summary_path}")

# ------------------------------------------------------------
# Final export summary
# ------------------------------------------------------------

print("\nEXPORT SUMMARY")
print("==============")

for path in [
    queue_path,
    monitoring_path,
    drift_path,
    reason_summary_path
]:
    print(f"✓ {path}")

print("\nW07 exports are ready for reuse in the paper.")

Saved action queue: work/outputs/content_action_queue.csv
Saved monitoring rules: work/outputs/monitoring_retrain_triggers.csv
Saved feature drift summary: work/outputs/feature_drift_summary.csv
Saved reason-code summary: work/outputs/reason_code_summary.csv

EXPORT SUMMARY
✓ work/outputs/content_action_queue.csv
✓ work/outputs/monitoring_retrain_triggers.csv
✓ work/outputs/feature_drift_summary.csv
✓ work/outputs/reason_code_summary.csv

W07 exports are ready for reuse in the paper.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.